# AUG-PE: Differentially Private Synthetic Text via Foundation Model APIs

This notebook walks through the AUG-PE algorithm step by step,
importing directly from the original codebase.

**Prerequisites**: Run `bash scripts/local_scripts/install_gpu.sh` first.

In [ ]:
import os, sys
import numpy as np
import collections

# Ensure repo root is on the path
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

print(f'Working directory: {os.getcwd()}')

## 1. Privacy Accounting

Before running any experiment, verify the DP noise multipliers from the paper.

In [ ]:
from src.dp_accounting import compute_sigma, compute_epsilon, compute_delta_default

# Yelp: n_priv = 1,939,290
n_priv = 1_939_290
delta = compute_delta_default(n_priv)
T = 10

print(f'Yelp: n_priv={n_priv:,}, delta={delta:.2e}, T={T}')
print()

for eps_target in [1.0, 2.0, 4.0]:
    sigma = compute_sigma(eps_target, T, delta)
    eps_check = compute_epsilon(sigma, T, delta)
    print(f'  epsilon={eps_target:.1f} => sigma={sigma:.2f} (verify: eps={eps_check:.4f})')

## 2. Load Private Data

Load the Yelp dataset and examine its label distribution.

In [ ]:
from src.dpsda.data_loader import load_data

train_data, train_labels, label_counter, label_indexer = load_data(
    dataset='yelp',
    data_file='data/yelp/train.csv',
    num_samples=5000,  # subsample for this demo
)

print(f'Loaded {len(train_data)} private samples')
print(f'Number of label combinations: {len(label_counter)}')
print()
print('Top 10 label combinations:')
for label, count in label_counter.most_common(10):
    print(f'  {label}: {count}')

In [ ]:
# Show a few examples
print('--- Sample private texts ---')
for i in range(3):
    print(f'\n[{train_labels[i]}]')
    print(train_data[i][:200] + '...' if len(train_data[i]) > 200 else train_data[i])

## 3. Compute Private Embeddings

Use the sentence-transformer model to embed the private data (Algorithm 1, Line 1).

In [ ]:
from src.dpsda.feature_extractor import extract_features

EMBEDDING_MODEL = 'stsb-roberta-base-v2'

print(f'Computing embeddings with {EMBEDDING_MODEL}...')
private_features = extract_features(
    data=train_data,
    batch_size=1024,
    model_name=EMBEDDING_MODEL,
)
print(f'Private embeddings shape: {private_features.shape}')

## 4. RANDOM_API: Generate Initial Synthetic Samples

Use GPT-2 with category/rating prompts to generate initial samples (Algorithm 1, Line 2).

In [ ]:
from src.apis.hf_api import HFAPI

# Instantiate the HuggingFace GPT-2 API
# Using small batch size and few samples for this demo
api = HFAPI(
    model_type='gpt2',
    variation_type='yelp_rephrase_tone',
    use_subcategory=True,
    output_dir=None,
    seed=42,
    mlm_probability=0.5,
    length=64,
    temperature=1.4,
    top_k=50,
    top_p=0.9,
    repetition_penalty=1.0,
    do_sample=True,
    fp16=True,
    no_cuda=False,
    random_sampling_batch_size=64,
    num_beams=5,
    dry_run=False,
    variation_batch_size=64,
)
print('GPT-2 model loaded.')

In [ ]:
# Generate a small set of initial samples (Nsyn_demo samples)
Nsyn_demo = 100  # small for demo; paper uses 5000

# Scale down the label counter proportionally
demo_counter = collections.Counter()
total = sum(label_counter.values())
for label, count in label_counter.items():
    demo_count = max(1, round(count / total * Nsyn_demo))
    demo_counter[label] = demo_count

print(f'Generating {sum(demo_counter.values())} initial samples...')
initial_samples, initial_labels, sync_counter, all_prompts = api.text_random_sampling(
    num_samples=Nsyn_demo,
    prompt_counter=label_counter,
)
print(f'Generated {len(initial_samples)} initial samples')
print()
print('--- Sample generated texts ---')
for i in range(min(3, len(initial_samples))):
    print(f'\n[{initial_labels[i]}]')
    text = initial_samples[i]
    print(text[:200] + '...' if len(text) > 200 else text)

## 5. One PE Iteration

Run the core loop of AUG-PE once: embed synthetic samples, compute DP histogram,
select top samples, generate variations.

In [ ]:
from src.dpsda.dp_counter import dp_nn_histogram

# Step 5a: Embed synthetic samples (K=0, self-embedding)
print('Computing synthetic embeddings...')
syn_features = extract_features(
    data=initial_samples,
    batch_size=1024,
    model_name=EMBEDDING_MODEL,
)
print(f'Synthetic embeddings shape: {syn_features.shape}')

In [ ]:
# Step 5b: DP Nearest Neighbor Histogram (one class at a time)
# For demo: use sigma=0 (non-private) to see clean signal
sigma = 0.0

private_classes = list(label_counter.keys())
all_counts = np.zeros(len(initial_samples))

current_idx = 0
for cls_label in private_classes:
    n_cls = sync_counter.get(cls_label, 0)
    if n_cls == 0:
        continue
    
    cls_syn_features = syn_features[current_idx:current_idx + n_cls]
    cls_pri_indices = label_indexer[cls_label]
    cls_pri_features = private_features[cls_pri_indices]
    
    count, clean_count = dp_nn_histogram(
        public_features=cls_syn_features,
        private_features=cls_pri_features,
        noise_multiplier=sigma,
    )
    all_counts[current_idx:current_idx + n_cls] = count
    current_idx += n_cls

print(f'Histogram: {len(all_counts)} bins, sum={all_counts.sum():.0f}')
print(f'Non-zero bins: {(all_counts > 0).sum()}')
print(f'Top 5 vote counts: {sorted(all_counts, reverse=True)[:5]}')

In [ ]:
# Step 5c: Rank-based selection (select top samples)
# For AUG-PE with L>1: select top Nsyn/L samples, then generate L-1 variations
L = 2  # small for demo; paper uses L=7
selected_size = len(initial_samples) // L

sort_indices = np.argsort(-all_counts)
selected_indices = sort_indices[:selected_size]

selected_samples = [initial_samples[i] for i in selected_indices]
selected_labels = [initial_labels[i] for i in selected_indices]

print(f'Selected {len(selected_samples)} samples (top by histogram votes)')
print(f'Vote range of selected: [{all_counts[selected_indices[-1]]:.0f}, {all_counts[selected_indices[0]]:.0f}]')

In [ ]:
# Step 5d: VARIATION_API - generate paraphrased variations
print(f'Generating {L-1} variation(s) per selected sample...')
variations, var_labels, _, _, _ = api.text_variation(
    sequences=selected_samples,
    additional_info=selected_labels,
    num_variations_per_sequence=L - 1,
    variation_degree=0.5,
)

print(f'Variations shape: {variations.shape}')
print()
print('--- Original vs Variation ---')
for i in range(min(2, len(selected_samples))):
    print(f'\nOriginal [{selected_labels[i]}]:')
    print(f'  {selected_samples[i][:150]}')
    print(f'Variation:')
    print(f'  {variations[i, 0][:150]}')

## 6. FID Measurement

Compute the Frechet Inception Distance between the synthetic and private embedding distributions.

In [ ]:
from src.dpsda.metrics import calculate_fid

# FID of initial samples vs private data
fid_initial = calculate_fid(syn_features, private_features)
print(f'FID (initial random samples vs private): {fid_initial:.2f}')

# FID of selected samples vs private data
selected_features = extract_features(
    data=selected_samples,
    batch_size=1024,
    model_name=EMBEDDING_MODEL,
)
fid_selected = calculate_fid(selected_features, private_features)
print(f'FID (after 1 PE iteration, selected): {fid_selected:.2f}')
print(f'FID improvement: {fid_initial - fid_selected:.2f}')

## 7. Next Steps

This demo showed one iteration of AUG-PE. For the full experiment:

```bash
# Precompute full embeddings
bash scripts/embeddings.sh --yelp

# Run full AUG-PE (20 iterations, GPT-2, Yelp, non-DP)
export CUDA_VISIBLE_DEVICES=0
bash scripts/hf/yelp/generate.sh

# Evaluate downstream accuracy
bash scripts/hf/yelp/downstream.sh
```

All source code lives under `src/`. See `src/config.py` for the paper's
hyperparameter defaults and `src/dp_accounting.py` for privacy budget calculations.